# BeetleCast 01 — Real-data baseline

This notebook creates the first working BeetleCast model from:

- the F3 Sentinel-2 Zarr cube;
- the cleaned bark-beetle polygons;
- a conservative satellite-derived forest mask.

It deliberately does **not** depend on AEF, Prithvi, Sentinel-1, or the large high-resolution scenes yet.

## Outputs

It writes the following into `outputs/baseline/`:

- `beetlecast_training_table.csv`
- `beetlecast_metrics.csv`
- `beetlecast_feature_importance.csv`
- `beetlecast_risk_2023.tif`
- `beetlecast_top10_inspection_zones.gpkg`
- `beetlecast_money_shot.png`

## Scientific framing

The 2022 and 2023 polygons come from different high-resolution source footprints, so this is an **early-damage risk baseline**, not yet a validated biological spread forecast.

The model uses satellite information from before each label year:

- 2022 labels ← features through 2021
- 2023 labels ← features through 2022


In [14]:
from pathlib import Path
import warnings
import json
import math

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import xarray as xr
import rioxarray  # registers the .rio accessor

from affine import Affine
from pyproj import CRS
from rasterio.features import rasterize
from rasterio.transform import from_origin
from rasterio.windows import Window, bounds as window_bounds
import rasterio
from shapely.geometry import box

from scipy.ndimage import binary_dilation

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.utils.class_weight import compute_sample_weight

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
PATCH_PIXELS = 5          # 5 x 10 m = approximately 50 m patches
NEGATIVE_RATIO = 3        # three forest controls per positive patch
POSITIVE_FRACTION = 0.05  # at least 5% of a patch intersects mapped damage
FOREST_NDVI_THRESHOLD = 0.45
COVERAGE_BUFFER_METRES = 2000
SPATIAL_BLOCK_PATCHES = 10
TEST_FRACTION = 0.25

cwd = Path.cwd()
if (cwd / "hackathon_data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "hackathon_data").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not find hackathon_data/. Launch Jupyter from the project root "
        "or place this notebook in project_root/notebooks/."
    )

DATA_ROOT = PROJECT_ROOT / "hackathon_data"
RAW_ROOT = DATA_ROOT / "raw"
PROCESSED_ROOT = DATA_ROOT / "processed"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "baseline"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)


PROJECT_ROOT: /Users/hemat/Desktop/hackathon-demo
OUTPUT_ROOT: /Users/hemat/Desktop/hackathon-demo/outputs/baseline


## 1. Locate and open the real inputs

In [15]:
# Sentinel-2 cube
zarr_paths = list(RAW_ROOT.rglob("cube.zarr"))
f3_zarr = [
    p for p in zarr_paths
    if "f3" in str(p).lower()
    or "bark" in str(p).lower()
    or "germany" in str(p).lower()
]
assert f3_zarr or zarr_paths, "No cube.zarr found under hackathon_data/raw"
CUBE_PATH = (f3_zarr or zarr_paths)[0]

# Prefer a cleaned label GeoPackage anywhere in the project.
label_paths = list(PROJECT_ROOT.rglob("F3_bark_beetle_labels_in_aoi.gpkg"))

# Otherwise find the original bark-beetle GeoJSON/GeoPackage.
if not label_paths:
    label_paths = [
        p for p in PROJECT_ROOT.rglob("*")
        if p.is_file()
        and p.suffix.lower() in {".gpkg", ".geojson"}
        and (
            "bark_beetle" in p.name.lower()
            or "bark_bettle" in p.name.lower()
            or ("bark" in p.name.lower() and "beetle" in p.name.lower())
            or ("bark" in p.name.lower() and "bettle" in p.name.lower())
        )
        and "aoi" not in p.name.lower()
    ]

assert label_paths, (
    "No bark-beetle label GeoJSON or GeoPackage found anywhere under the project root. "
    "Put F3_bark_beetle_locations.geojson under hackathon_data/raw/ "
    "or put F3_bark_beetle_labels_in_aoi.gpkg anywhere in the project."
)

LABEL_PATH = label_paths[0]

print("Sentinel-2:", CUBE_PATH)
print("Labels:", LABEL_PATH)

ds = xr.open_zarr(CUBE_PATH, chunks="auto")

if LABEL_PATH.suffix.lower() == ".gpkg":
    try:
        labels = gpd.read_file(LABEL_PATH, layer="bark_beetle_labels")
    except Exception:
        labels = gpd.read_file(LABEL_PATH)
else:
    labels = gpd.read_file(LABEL_PATH)

# Add label year when loading the original file.
if "label_year" not in labels.columns and "tile_name" in labels.columns:
    labels["label_year"] = (
        labels["tile_name"]
        .astype(str)
        .str.extract(r"_rp_(20\d{2})_")[0]
        .astype("Int64")
    )

# If the file is the original full label layer, clip it to the AOI and save a clean copy.
if "in_aoi" not in LABEL_PATH.stem.lower():
    aoi_candidates = [
        p for p in PROJECT_ROOT.rglob("*.geojson")
        if "aoi" in p.name.lower()
        and ("f3" in p.name.lower() or "germany" in p.name.lower() or "bark" in p.name.lower())
    ]

    if aoi_candidates:
        aoi = gpd.read_file(aoi_candidates[0])
        labels = labels.to_crs(aoi.crs)
        labels["geometry"] = labels.geometry.make_valid()
        labels = gpd.clip(labels, aoi)
        labels = labels[~labels.geometry.is_empty].copy()

        cleaned_dir = PROCESSED_ROOT / "labels"
        cleaned_dir.mkdir(parents=True, exist_ok=True)
        cleaned_path = cleaned_dir / "F3_bark_beetle_labels_in_aoi.gpkg"

        labels.to_file(
            cleaned_path,
            layer="bark_beetle_labels",
            driver="GPKG",
        )

        LABEL_PATH = cleaned_path
        print("Created cleaned labels:", cleaned_path)
    else:
        print(
            "Warning: original label file loaded, but no F3 AOI GeoJSON was found. "
            "Continuing with the full label layer."
        )

print(ds)
print("\nLabel rows:", len(labels))
print("Label years:")
print(labels["label_year"].value_counts(dropna=False).sort_index())


Sentinel-2: /Users/hemat/Desktop/hackathon-demo/hackathon_data/raw/sentinal2/sentinel2/F3_Germany_BarkBeetle/2017-01-01_2025-12-31/cube.zarr
Labels: /Users/hemat/Desktop/hackathon-demo/hackathon_data/clean/F3_bark_beetle_labels_in_aoi.gpkg
<xarray.Dataset> Size: 3GB
Dimensions:      (time: 108, y: 1262, x: 1065)
Coordinates:
  * time         (time) datetime64[ns] 864B 2017-01-01 2017-02-01 ... 2025-12-01
  * y            (y) float64 10kB 5.512e+06 5.512e+06 ... 5.499e+06 5.499e+06
  * x            (x) float64 9kB 3.546e+05 3.546e+05 ... 3.652e+05 3.652e+05
    spatial_ref  int64 8B ...
Data variables:
    B02          (time, y, x) uint16 290MB dask.array<chunksize=(49, 1262, 1065), meta=np.ndarray>
    B03          (time, y, x) uint16 290MB dask.array<chunksize=(49, 1262, 1065), meta=np.ndarray>
    B04          (time, y, x) uint16 290MB dask.array<chunksize=(49, 1262, 1065), meta=np.ndarray>
    B05          (time, y, x) uint16 290MB dask.array<chunksize=(49, 1262, 1065), meta=np.ndar

## 2. Standardise grid orientation, CRS, and transform

In [16]:
# Keep raster rows north-to-south.
if float(ds.y.values[0]) < float(ds.y.values[-1]):
    ds = ds.sortby("y", ascending=False)

x = np.asarray(ds.x.values)
y = np.asarray(ds.y.values)

dx = float(abs(x[1] - x[0]))
dy = float(abs(y[1] - y[0]))

left = float(x.min() - dx / 2)
top = float(y.max() + dy / 2)
transform = from_origin(left, top, dx, dy)

def infer_crs(dataset):
    try:
        if dataset.rio.crs is not None:
            return CRS.from_user_input(dataset.rio.crs)
    except Exception:
        pass

    if "spatial_ref" in dataset:
        attrs = dataset["spatial_ref"].attrs
        for key in ("crs_wkt", "spatial_ref", "wkt"):
            value = attrs.get(key)
            if value:
                return CRS.from_user_input(value)

    raise ValueError(
        "Could not infer the Sentinel-2 CRS. Inspect ds['spatial_ref'].attrs."
    )

grid_crs = infer_crs(ds)

print("Grid CRS:", grid_crs)
print("Pixel size:", dx, "x", dy)
print("Transform:", transform)

labels_grid = labels.to_crs(grid_crs)
labels_grid["geometry"] = labels_grid.geometry.make_valid()


Grid CRS: PROJCS["WGS 84 / UTM zone 32N",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",9],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","32632"]]
Pixel size: 10.0 x 10.0
Transform: | 10.00, 0.00, 354585.00|
| 0.00,-10.00, 5512065.00|
| 0.00, 0.00, 1.00|


## 3. Harmonise reflectance and build temporal features

For each label year, the feature cutoff is the end of the previous calendar year.

Features include:

- recent median NDVI, NDRE, and NBR;
- prior-year medians;
- year-over-year changes;
- recent temporal variability;
- fraction of valid observations.


In [17]:
required_bands = {"B04", "B05", "B08", "B12"}
missing = required_bands - set(ds.data_vars)
if missing:
    raise KeyError(f"Missing required bands: {missing}")

def clean_reflectance(dataset, band):
    values = dataset[band].astype("float32")

    if "n_obs" in dataset:
        values = values.where(dataset["n_obs"] > 0)

    cutoff = np.datetime64("2022-01-25")
    values = xr.where(
        dataset.time >= cutoff,
        xr.where(values >= 1000, values - 1000, 0),
        values,
    )
    return values / 10000.0

red = clean_reflectance(ds, "B04")
red_edge = clean_reflectance(ds, "B05")
nir = clean_reflectance(ds, "B08")
swir2 = clean_reflectance(ds, "B12")

ndvi = (nir - red) / (nir + red)
ndre = (nir - red_edge) / (nir + red_edge)
nbr = (nir - swir2) / (nir + swir2)

def safe_window(da, start, end, name):
    subset = da.sel(time=slice(start, end))
    if subset.sizes.get("time", 0) == 0:
        raise ValueError(f"No observations in {name}: {start} to {end}")
    return subset

def make_feature_stack(label_year):
    recent_start = f"{label_year - 1}-01-01"
    recent_end = f"{label_year - 1}-12-31"
    prior_start = f"{label_year - 2}-01-01"
    prior_end = f"{label_year - 2}-12-31"

    recent_ndvi = safe_window(ndvi, recent_start, recent_end, "recent NDVI")
    recent_ndre = safe_window(ndre, recent_start, recent_end, "recent NDRE")
    recent_nbr = safe_window(nbr, recent_start, recent_end, "recent NBR")

    prior_ndvi = safe_window(ndvi, prior_start, prior_end, "prior NDVI")
    prior_ndre = safe_window(ndre, prior_start, prior_end, "prior NDRE")
    prior_nbr = safe_window(nbr, prior_start, prior_end, "prior NBR")

    recent_valid = xr.where(np.isfinite(recent_ndvi), 1.0, 0.0).mean("time")

    features = xr.Dataset({
        "ndvi_recent": recent_ndvi.median("time", skipna=True),
        "ndre_recent": recent_ndre.median("time", skipna=True),
        "nbr_recent": recent_nbr.median("time", skipna=True),
        "ndvi_prior": prior_ndvi.median("time", skipna=True),
        "ndre_prior": prior_ndre.median("time", skipna=True),
        "nbr_prior": prior_nbr.median("time", skipna=True),
        "ndvi_change": (
            recent_ndvi.median("time", skipna=True)
            - prior_ndvi.median("time", skipna=True)
        ),
        "ndre_change": (
            recent_ndre.median("time", skipna=True)
            - prior_ndre.median("time", skipna=True)
        ),
        "nbr_change": (
            recent_nbr.median("time", skipna=True)
            - prior_nbr.median("time", skipna=True)
        ),
        "ndvi_std": recent_ndvi.std("time", skipna=True),
        "ndre_std": recent_ndre.std("time", skipna=True),
        "valid_fraction": recent_valid,
    })

    # Aggregate to approximately 50 m patches before loading into memory.
    patch_features = features.coarsen(
        y=PATCH_PIXELS,
        x=PATCH_PIXELS,
        boundary="trim",
    ).mean(skipna=True)

    return patch_features.compute()


## 4. Rasterise labels and create forest-only patch samples

Because the label layer contains positive damage polygons but not verified healthy polygons:

- positives are patches intersecting mapped damage;
- controls are sampled from forest-like patches in the same broad source-year vicinity;
- patches immediately adjacent to positives are excluded from controls;
- a conservative multi-year NDVI mask is used until a dedicated forest layer is added.


In [18]:
height = ds.sizes["y"]
width = ds.sizes["x"]

def rasterise_geometries(gdf, fill=0, default_value=1, dtype="uint8"):
    shapes = [
        (geom, default_value)
        for geom in gdf.geometry
        if geom is not None and not geom.is_empty
    ]
    return rasterize(
        shapes,
        out_shape=(height, width),
        transform=transform,
        fill=fill,
        dtype=dtype,
        all_touched=True,
    )

def make_patch_table(label_year):
    print(f"\nBuilding cohort {label_year}...")

    feature_ds = make_feature_stack(label_year)

    cohort_labels = labels_grid[
        labels_grid["label_year"].astype("Int64") == label_year
    ].copy()
    if cohort_labels.empty:
        raise ValueError(f"No labels found for {label_year}")

    # Positive-polygon fraction per patch.
    positive_pixel = rasterise_geometries(cohort_labels)
    positive_da = xr.DataArray(
        positive_pixel,
        dims=("y", "x"),
        coords={"y": ds.y, "x": ds.x},
    )
    positive_fraction = positive_da.coarsen(
        y=PATCH_PIXELS,
        x=PATCH_PIXELS,
        boundary="trim",
    ).mean().values

    # Approximate source coverage: buffered convex hull around this year's labels.
    hull = cohort_labels.geometry.union_all().convex_hull
    buffer_distance = COVERAGE_BUFFER_METRES if grid_crs.is_projected else 0.02
    coverage_geom = hull.buffer(buffer_distance)

    coverage_pixel = rasterize(
        [(coverage_geom, 1)],
        out_shape=(height, width),
        transform=transform,
        fill=0,
        dtype="uint8",
        all_touched=True,
    )
    coverage_da = xr.DataArray(
        coverage_pixel,
        dims=("y", "x"),
        coords={"y": ds.y, "x": ds.x},
    )
    coverage_patch = coverage_da.coarsen(
        y=PATCH_PIXELS,
        x=PATCH_PIXELS,
        boundary="trim",
    ).mean().values >= 0.5

    # Forest proxy: vegetation was forest-like in either recent or prior year.
    forest_patch = (
        np.maximum(
            feature_ds["ndvi_recent"].values,
            feature_ds["ndvi_prior"].values,
        ) >= FOREST_NDVI_THRESHOLD
    )
    forest_patch &= feature_ds["valid_fraction"].values >= 0.25

    positive_mask = positive_fraction >= POSITIVE_FRACTION

    # Exclude immediate neighbours of positives from the negative pool.
    exclusion_zone = binary_dilation(positive_mask, iterations=2)

    negative_pool = (
        forest_patch
        & coverage_patch
        & (~exclusion_zone)
        & np.isfinite(feature_ds["ndvi_recent"].values)
    )

    pos_rows, pos_cols = np.where(positive_mask & forest_patch & coverage_patch)
    neg_rows, neg_cols = np.where(negative_pool)

    if len(pos_rows) == 0:
        raise ValueError(f"No positive forest patches for {label_year}")
    if len(neg_rows) == 0:
        raise ValueError(f"No negative candidates for {label_year}")

    rng = np.random.default_rng(RANDOM_STATE + label_year)
    n_negative = min(len(neg_rows), len(pos_rows) * NEGATIVE_RATIO)
    chosen = rng.choice(len(neg_rows), size=n_negative, replace=False)

    sample_rows = np.concatenate([pos_rows, neg_rows[chosen]])
    sample_cols = np.concatenate([pos_cols, neg_cols[chosen]])
    target = np.concatenate([
        np.ones(len(pos_rows), dtype=int),
        np.zeros(n_negative, dtype=int),
    ])

    frame = pd.DataFrame({
        "label_year": label_year,
        "patch_row": sample_rows,
        "patch_col": sample_cols,
        "target": target,
        "positive_fraction": positive_fraction[sample_rows, sample_cols],
    })

    for variable in feature_ds.data_vars:
        frame[variable] = feature_ds[variable].values[sample_rows, sample_cols]

    # Spatial block identifier for leakage-resistant splitting.
    frame["spatial_block"] = (
        frame["label_year"].astype(str)
        + "_"
        + (frame["patch_row"] // SPATIAL_BLOCK_PATCHES).astype(str)
        + "_"
        + (frame["patch_col"] // SPATIAL_BLOCK_PATCHES).astype(str)
    )

    metadata = {
        "label_year": label_year,
        "feature_ds": feature_ds,
        "positive_fraction": positive_fraction,
        "forest_patch": forest_patch,
        "coverage_patch": coverage_patch,
    }

    print("Positive patches:", int((frame.target == 1).sum()))
    print("Control patches:", int((frame.target == 0).sum()))
    return frame, metadata

cohort_frames = []
cohort_metadata = {}

for year in sorted(labels["label_year"].dropna().astype(int).unique()):
    frame, metadata = make_patch_table(year)
    cohort_frames.append(frame)
    cohort_metadata[year] = metadata

training = pd.concat(cohort_frames, ignore_index=True)
print("\nCombined training rows:", len(training))
print(training.groupby(["label_year", "target"]).size())



Building cohort 2022...
Positive patches: 2534
Control patches: 7602

Building cohort 2023...
Positive patches: 2452
Control patches: 7356

Combined training rows: 19944
label_year  target
2022        0         7602
            1         2534
2023        0         7356
            1         2452
dtype: int64


## 5. Save and inspect the training table

In [19]:
feature_columns = [
    "ndvi_recent",
    "ndre_recent",
    "nbr_recent",
    "ndvi_prior",
    "ndre_prior",
    "nbr_prior",
    "ndvi_change",
    "ndre_change",
    "nbr_change",
    "ndvi_std",
    "ndre_std",
    "valid_fraction",
]

training_path = OUTPUT_ROOT / "beetlecast_training_table.csv"
training.to_csv(training_path, index=False)

display(training.head())
display(training.groupby(["label_year", "target"])[feature_columns].mean())

print("Saved:", training_path)


,label_year,patch_row,patch_col,target,positive_fraction,ndvi_recent,ndre_recent,nbr_recent,ndvi_prior,ndre_prior,nbr_prior,ndvi_change,ndre_change,nbr_change,ndvi_std,ndre_std,valid_fraction,spatial_block
0,2022,29,189,1,0.20,0.549950,0.355150,0.299444,0.664222,0.408813,0.450751,-0.114271,-0.053662,-0.151308,0.174343,0.130195,0.833333,2022_2_18
1,2022,29,190,1,0.28,0.632780,0.436650,0.398590,0.653001,0.428360,0.504165,-0.020221,0.008289,-0.105576,0.167086,0.132359,0.833333,2022_2_19
2,2022,30,184,1,0.08,0.757545,0.554262,0.611344,0.808004,0.570768,0.692338,-0.050459,-0.016506,-0.080994,0.138427,0.110972,0.896667,2022_3_18
3,2022,30,186,1,0.24,0.703184,0.469116,0.452635,0.750894,0.495153,0.580500,-0.047711,-0.026037,-0.127866,0.165656,0.126585,0.840000,2022_3_18
4,2022,30,187,1,0.32,0.724626,0.454277,0.400069,0.686882,0.448442,0.479371,0.037743,0.005835,-0.079302,0.150354,0.112018,0.833333,2022_3_18


ndvi_recent  ndre_recent  nbr_recent  ndvi_prior  \
label_year target                                                     
2022       0          0.705607     0.482355    0.508225    0.715065   
           1          0.666159     0.448234    0.422843    0.729776   
2023       0          0.745308     0.507860    0.570105    0.709783   
           1          0.707802     0.484811    0.508581    0.717125   

                   ndre_prior  nbr_prior  ndvi_change  ndre_change  \
label_year target                                                    
2022       0         0.483190   0.546611    -0.009458    -0.000835   
           1         0.498645   0.551341    -0.063617    -0.050411   
2023       0         0.484630   0.513815     0.035525     0.023230   
           1         0.495371   0.509807    -0.009324    -0.010559   

                   nbr_change  ndvi_std  ndre_std  valid_fraction  
label_year target                                                  
2022       0        -0.038386  0.182095  0.149492        0.870496  
           1        -0.128498  0.154179  0.122489        0.858891  
2023       0         0.056290  0.169468  0.137717        0.962440  
           1        -0.001225  0.143391  0.113208        0.959271

Saved: /Users/hemat/Desktop/hackathon-demo/outputs/baseline/beetlecast_training_table.csv


## 6. Spatially blocked train/test split

Neighbouring patches are kept in the same split. This avoids an unrealistically easy random-pixel test.


In [20]:
usable = training.dropna(subset=feature_columns).copy()

X = usable[feature_columns]
y = usable["target"].astype(int)
groups = usable["spatial_block"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=TEST_FRACTION,
    random_state=RANDOM_STATE,
)
train_idx, test_idx = next(splitter.split(X, y, groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print("Train positive rate:", y_train.mean())
print("Test positive rate:", y_test.mean())
print("Train blocks:", groups.iloc[train_idx].nunique())
print("Test blocks:", groups.iloc[test_idx].nunique())


Train rows: 14741
Test rows: 5203
Train positive rate: 0.23207380774709993
Test positive rate: 0.30078800691908514
Train blocks: 474
Test blocks: 158


## 7. Train the floor model

In [ ]:
model = HistGradientBoostingClassifier(
    learning_rate=0.06,
    max_iter=300,
    max_leaf_nodes=31,
    min_samples_leaf=20,
    l2_regularization=1.0,
    random_state=RANDOM_STATE,
)

sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)
model.fit(X_train, y_train, sample_weight=sample_weight)

test_risk = model.predict_proba(X_test)[:, 1]
test_pred = (test_risk >= 0.5).astype(int)

def top_fraction_capture(y_true, risk, fraction=0.10):
    y_true = np.asarray(y_true).astype(int)
    risk = np.asarray(risk)
    n_select = max(1, int(np.ceil(len(risk) * fraction)))
    selected = np.argsort(risk)[-n_select:]
    total_positive = y_true.sum()
    if total_positive == 0:
        return np.nan
    return y_true[selected].sum() / total_positive

metrics = {
    "roc_auc": roc_auc_score(y_test, test_risk),
    "pr_auc": average_precision_score(y_test, test_risk),
    "precision_at_0.5": precision_score(y_test, test_pred, zero_division=0),
    "recall_at_0.5": recall_score(y_test, test_pred, zero_division=0),
    "f1_at_0.5": f1_score(y_test, test_pred, zero_division=0),
    "top_10_percent_capture": top_fraction_capture(y_test, test_risk, 0.10),
}

metrics_df = pd.DataFrame([metrics])
metrics_path = OUTPUT_ROOT / "beetlecast_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)

display(metrics_df.T.rename(columns={0: "value"}))
print("Saved:", metrics_path)


## 8. Permutation feature importance

In [ ]:
importance = permutation_importance(
    model,
    X_test,
    y_test,
    scoring="average_precision",
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

importance_df = pd.DataFrame({
    "feature": feature_columns,
    "importance_mean": importance.importances_mean,
    "importance_std": importance.importances_std,
}).sort_values("importance_mean", ascending=False)

importance_path = OUTPUT_ROOT / "beetlecast_feature_importance.csv"
importance_df.to_csv(importance_path, index=False)

display(importance_df)

ax = importance_df.sort_values("importance_mean").plot(
    x="feature",
    y="importance_mean",
    kind="barh",
    figsize=(8, 6),
    legend=False,
)
ax.set_title("BeetleCast baseline — permutation importance")
ax.set_xlabel("Decrease in PR-AUC")
plt.tight_layout()
plt.show()

print("Saved:", importance_path)


## 9. Generate the 2023 risk map

The latest cohort is used for the first demo map. Predictions are produced only within the conservative forest and source-coverage masks.


In [ ]:
map_year = max(cohort_metadata)
meta = cohort_metadata[map_year]
feature_ds = meta["feature_ds"]

map_frame = pd.DataFrame({
    name: feature_ds[name].values.ravel()
    for name in feature_columns
})

eligible = (
    meta["forest_patch"]
    & meta["coverage_patch"]
    & np.all(np.isfinite(map_frame[feature_columns].values), axis=1).reshape(
        meta["forest_patch"].shape
    )
)

risk_grid = np.full(meta["forest_patch"].shape, np.nan, dtype="float32")
eligible_flat = eligible.ravel()

risk_grid.ravel()[eligible_flat] = model.predict_proba(
    map_frame.loc[eligible_flat, feature_columns]
)[:, 1].astype("float32")

patch_transform = transform * Affine.scale(PATCH_PIXELS, PATCH_PIXELS)
risk_tif = OUTPUT_ROOT / f"beetlecast_risk_{map_year}.tif"

with rasterio.open(
    risk_tif,
    "w",
    driver="GTiff",
    height=risk_grid.shape[0],
    width=risk_grid.shape[1],
    count=1,
    dtype="float32",
    crs=grid_crs,
    transform=patch_transform,
    nodata=np.nan,
    compress="deflate",
) as dst:
    dst.write(risk_grid, 1)

print("Saved:", risk_tif)


## 10. Export the top 10% inspection zones

In [ ]:
valid_risk = risk_grid[np.isfinite(risk_grid)]
threshold = np.quantile(valid_risk, 0.90)
top_mask = np.isfinite(risk_grid) & (risk_grid >= threshold)

rows, cols = np.where(top_mask)
records = []

for row, col in zip(rows, cols):
    left_b, bottom_b, right_b, top_b = window_bounds(
        Window(col, row, 1, 1),
        patch_transform,
    )
    records.append({
        "label_year": map_year,
        "risk": float(risk_grid[row, col]),
        "geometry": box(left_b, bottom_b, right_b, top_b),
    })

top_zones = gpd.GeoDataFrame(records, crs=grid_crs)
top_zones_path = OUTPUT_ROOT / "beetlecast_top10_inspection_zones.gpkg"
top_zones.to_file(top_zones_path, layer="top10_zones", driver="GPKG")

print("Top-zone patches:", len(top_zones))
print("Risk threshold:", threshold)
print("Saved:", top_zones_path)


## 11. Create the money-shot figure

In [ ]:
observed = meta["positive_fraction"]

fig, axes = plt.subplots(1, 3, figsize=(17, 6))

im0 = axes[0].imshow(observed, vmin=0, vmax=max(0.25, np.nanpercentile(observed, 99)))
axes[0].set_title(f"Mapped canopy damage\n{map_year} high-resolution labels")
axes[0].axis("off")
fig.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(risk_grid, vmin=0, vmax=1)
axes[1].set_title("BeetleCast risk\npre-label Sentinel-2 only")
axes[1].axis("off")
fig.colorbar(im1, ax=axes[1], fraction=0.046)

axes[2].imshow(risk_grid, vmin=0, vmax=1)
axes[2].contour(top_mask.astype(int), levels=[0.5], linewidths=1.0)
axes[2].contour(
    (observed >= POSITIVE_FRACTION).astype(int),
    levels=[0.5],
    linewidths=0.8,
)
axes[2].set_title(
    "Inspection frontier\n"
    f"Top-10% test capture: {metrics['top_10_percent_capture']:.1%}"
)
axes[2].axis("off")

plt.suptitle(
    "BeetleCast baseline: medium-resolution satellite risk vs mapped damage",
    fontsize=15,
)
plt.tight_layout()

money_shot = OUTPUT_ROOT / "beetlecast_money_shot.png"
plt.savefig(money_shot, dpi=220, bbox_inches="tight")
plt.show()

print("Saved:", money_shot)


## 12. Final report

In [ ]:
print("BASELINE COMPLETE")
print("-----------------")
print("Training rows:", len(usable))
print("Spatially blocked test rows:", len(X_test))
print(f"PR-AUC: {metrics['pr_auc']:.3f}")
print(f"ROC-AUC: {metrics['roc_auc']:.3f}")
print(f"Top-10% capture: {metrics['top_10_percent_capture']:.1%}")
print()
print("Outputs:")
for path in sorted(OUTPUT_ROOT.iterdir()):
    print("-", path.name)

aef_candidates = [
    p for p in RAW_ROOT.rglob("*.tif")
    if "aef" in str(p).lower() or "embedding" in str(p).lower()
]
print()
print("AEF TIFFs currently visible:", len(aef_candidates))
if aef_candidates:
    print("Next upgrade: add AEF bands and rerun the same blocked evaluation.")
else:
    print("Next upgrade: copy the F3 AEF annual TIFFs into hackathon_data/raw/aef_embeddings/.")
